In [0]:
%sql
CREATE CATALOG IF NOT EXISTS data_marts;

In [0]:
%sql
USE SCHEMA DEFAULT;

CREATE VOLUME IF NOT EXISTS landing;

In [0]:
%sql
USE CATALOG data_marts;

CREATE SCHEMA IF NOT EXISTS bronze;

In [0]:
from pyspark.sql import functions as F

In [0]:
def ingest_csv(nome_arquivo, nome_tabela):

    try:
        table_name = nome_tabela
        landing_path = f"/Volumes/data_marts/default/landing/{nome_arquivo}"

        df = spark.read.csv(landing_path, header=True, inferSchema=True)
        
        if df.count() == 0:
            print(f"O arquivo {nome_arquivo} está vazio ou não foi lido corretamente.")
          
        df_with_metadata = df.withColumn("timestamp_ingestion", F.current_timestamp())

        df_with_metadata.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{bronze_db_name}.{table_name}")

        print(f"✅ Tabela bronze.{nome_tabela} criada com sucesso\n")

    except Exception as E:
        print(f"Falha ao processor o arquivo {nome_tabela}: {str(E)}")

In [0]:
def ingest_json(dados_json, nome_tabela):

    try:

        table_name = nome_tabela

        if not dados_json:
            print(f" Sem dados para ingerir na tabela {table_name}.")
            return
        
        df = spark.createDataFrame(dados_json)

        df_with_metadata = df.withColumn("timestamp_ingestion", F.current_timestamp()).withColumn("fonte_dado", F.lit("API Banco Central"))

        df_with_metadata.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{bronze_db_name}.{table_name}")

        print(f"✅ Tabela bronze.{nome_tabela} criada com sucesso\n")

    except Exception as E:
        print(f"Falha ao processor o arquivo {nome_tabela}: {str(E)}")

In [0]:
import requests

def pegar_cotacao(data_inicio, data_fim):

    url = (
        f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
        f"CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?"
        f"@dataInicial='{data_inicio}'&@dataFinalCotacao='{data_fim}'"
        f"&$select=dataHoraCotacao,cotacaoCompra&$format=json"
    )

    response = requests.get(url)
    
    dados = response.json().get('value', [])
    
    return dados


In [0]:
catalogo = "data_marts"
bronze_db_name = "bronze"

In [0]:
ingest_csv("olist_customers_dataset.csv", "tb_customers")
ingest_csv("olist_geolocation_dataset.csv", "tb_geolocalizacao")
ingest_csv("olist_order_items_dataset.csv", "tb_order_items")
ingest_csv("olist_order_payments_dataset.csv", "tb_order_payments")
ingest_csv("olist_order_reviews_dataset.csv", "tb_order_reviews")
ingest_csv("olist_orders_dataset.csv", "tb_orders")
ingest_csv("olist_products_dataset.csv", "tb_products")
ingest_csv("olist_sellers_dataset.csv", "tb_sellers")
ingest_csv("product_category_name_translation.csv", "tb_product_category_name_translation")

In [0]:
df_order = spark.table("data_marts.bronze.tb_orders")

In [0]:
df_order_min = df_order.select(F.min("order_purchase_timestamp"))
df_order_min.show()

In [0]:
df_order_max = df_order.select(F.max("order_purchase_timestamp"))
df_order_max.show()

In [0]:
dbutils.widgets.text("data_inicio", "09-01-2016", "Data Início (MM-DD-AAAA)")

dbutils.widgets.text("data_fim",    "10-30-2018", "Data Fim (MM-DD-AAAA)")

In [0]:
data_ini = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

dados_brutos = pegar_cotacao(data_ini, data_fim)

ingest_json(dados_brutos, "tb_cotacao_dolar")